# 📈 Analyse des Séries Temporelles — Bitcoin Historical Data

**Projet 6 — Data Science | Prévision des tendances temporelles**

---

## Objectif
Ce notebook réalise une analyse complète des séries temporelles sur les données historiques du Bitcoin :
- **EDA** : exploration, nettoyage, feature engineering
- **Modèles classiques** : ARIMA, SARIMA
- **Modèles modernes** : Prophet (Facebook), XGBoost
- **Deep Learning** : LSTM (Long Short-Term Memory)
- **Évaluation** : comparaison des modèles (MAE, RMSE, MAPE)

---

### Sections
1. Chargement & aperçu des données
2. Nettoyage & feature engineering
3. Analyse exploratoire (EDA)
4. Stationnarité & décomposition
5. Modèle ARIMA
6. Modèle SARIMA
7. Modèle Prophet
8. Modèle XGBoost
9. Modèle LSTM
10. Comparaison des modèles & insights

---
## 0. Imports & Configuration

In [ ]:
# ── Standard ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
import os

# ── Stats ──────────────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import pmdarima as pm

# ── Prophet ────────────────────────────────────────────────────────────────────
from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly

# ── XGBoost ────────────────────────────────────────────────────────────────────
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Deep Learning (LSTM) ───────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── Affichage ──────────────────────────────────────────────────────────────────
plt.style.use('dark_background')
PALETTE = ['#F7931A', '#00C4FF', '#FF6B6B', '#4ECDC4', '#A8E6CF']
sns.set_palette(PALETTE)

os.makedirs('../plots', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print('✅ Imports OK')
print(f'   TensorFlow : {tf.__version__}')
print(f'   XGBoost    : {xgb.__version__}')

---
## 1. Chargement & Aperçu des Données

In [ ]:
# ── Chargement ─────────────────────────────────────────────────────────────────
# Dataset : https://www.kaggle.com/datasets/mczielinski/bitcoin-historical-data
# Colonnes : Timestamp, Open, High, Low, Close, Volume_(BTC), Volume_(Currency), Weighted_Price

df_raw = pd.read_csv('../data/bitstampUSD_1-min_data_2012-01-01_to_2021-03-31.csv')
print('Shape brut :', df_raw.shape)
df_raw.head()

In [ ]:
# ── Audit qualité ─────────────────────────────────────────────────────────────
print('=== Informations générales ===')
df_raw.info()
print('\n=== Valeurs manquantes ===')
print(df_raw.isnull().sum())
print('\n=== Statistiques descriptives ===')
df_raw.describe()

---
## 2. Nettoyage & Feature Engineering

In [ ]:
# ── Conversion timestamp → datetime ──────────────────────────────────────────
df = df_raw.copy()
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Timestamp', inplace=True)
df.sort_index(inplace=True)

# ── Rééchantillonnage journalier (médiane) ───────────────────────────────────
df_daily = df.resample('D').agg({
    'Open'             : 'first',
    'High'             : 'max',
    'Low'              : 'min',
    'Close'            : 'last',
    'Volume_(BTC)'     : 'sum',
    'Volume_(Currency)': 'sum',
    'Weighted_Price'   : 'mean'
})

# ── Suppression des lignes entièrement nulles (NaN BTC manquant) ─────────────
df_daily.dropna(how='all', inplace=True)

# ── Interpolation linéaire pour les valeurs manquantes résiduelles ───────────
df_daily.interpolate(method='linear', inplace=True)

print('✅ Données journalières :', df_daily.shape)
print(f'   Période : {df_daily.index.min().date()} → {df_daily.index.max().date()}')
df_daily.head()

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────

df_daily['Returns']       = df_daily['Close'].pct_change()           # rendement journalier
df_daily['Log_Returns']   = np.log(df_daily['Close'] / df_daily['Close'].shift(1))
df_daily['MA_7']          = df_daily['Close'].rolling(7).mean()      # moyenne mobile 7j
df_daily['MA_30']         = df_daily['Close'].rolling(30).mean()     # moyenne mobile 30j
df_daily['MA_90']         = df_daily['Close'].rolling(90).mean()     # moyenne mobile 90j
df_daily['Volatility_30'] = df_daily['Returns'].rolling(30).std()    # volatilité 30j
df_daily['Range']         = df_daily['High'] - df_daily['Low']       # amplitude H-L
df_daily['Year']          = df_daily.index.year
df_daily['Month']         = df_daily.index.month
df_daily['DayOfWeek']     = df_daily.index.dayofweek
df_daily['Quarter']       = df_daily.index.quarter

# Bandes de Bollinger (20j ± 2σ)
df_daily['BB_Mid']   = df_daily['Close'].rolling(20).mean()
df_daily['BB_Upper'] = df_daily['BB_Mid'] + 2 * df_daily['Close'].rolling(20).std()
df_daily['BB_Lower'] = df_daily['BB_Mid'] - 2 * df_daily['Close'].rolling(20).std()

# RSI (Relative Strength Index 14j)
delta = df_daily['Close'].diff()
gain  = delta.where(delta > 0, 0).rolling(14).mean()
loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs    = gain / loss
df_daily['RSI'] = 100 - (100 / (1 + rs))

print(f'✅ {len(df_daily.columns)} colonnes après feature engineering')
df_daily.tail()

---
## 3. Analyse Exploratoire (EDA)

In [ ]:
# ── 3.1 Prix historique complet ───────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Bitcoin — Analyse Historique Complète', fontsize=18, fontweight='bold', color='#F7931A')

# Prix + moyennes mobiles
ax = axes[0]
ax.plot(df_daily.index, df_daily['Close'], color='#F7931A', lw=1, label='Prix Close', alpha=0.9)
ax.plot(df_daily.index, df_daily['MA_30'], color='#00C4FF', lw=1.5, label='MA 30j', alpha=0.8)
ax.plot(df_daily.index, df_daily['MA_90'], color='#FF6B6B', lw=1.5, label='MA 90j', alpha=0.8)
ax.fill_between(df_daily.index, df_daily['BB_Lower'], df_daily['BB_Upper'], alpha=0.1, color='#F7931A')
ax.set_ylabel('Prix (USD)', color='white')
ax.legend(loc='upper left')
ax.set_title('Prix de Clôture + Moyennes Mobiles + Bollinger')

# Volume
ax2 = axes[1]
ax2.bar(df_daily.index, df_daily['Volume_(Currency)'], color='#4ECDC4', alpha=0.6, width=1)
ax2.set_ylabel('Volume ($)', color='white')
ax2.set_title('Volume de Trading (USD)')

# Rendements journaliers
ax3 = axes[2]
returns = df_daily['Returns'].dropna()
colors  = ['#4ECDC4' if r >= 0 else '#FF6B6B' for r in returns]
ax3.bar(returns.index, returns, color=colors, alpha=0.7, width=1)
ax3.axhline(0, color='white', lw=0.5)
ax3.set_ylabel('Rendement', color='white')
ax3.set_title('Rendements Journaliers')

plt.tight_layout()
plt.savefig('../plots/01_historique_complet.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.2 Distribution des rendements ──────────────────────────────────────────
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution des Rendements Bitcoin', fontsize=15, color='#F7931A')

r = df_daily['Returns'].dropna()

# Histogramme
axes[0].hist(r, bins=100, color='#F7931A', alpha=0.7, edgecolor='none')
axes[0].set_title('Distribution des rendements')
axes[0].axvline(r.mean(), color='#00C4FF', lw=2, label=f'Moyenne : {r.mean():.4f}')
axes[0].legend()

# QQ Plot
stats.probplot(r, dist='norm', plot=axes[1])
axes[1].set_title('QQ-Plot (normalité ?)')
axes[1].get_lines()[0].set(color='#F7931A', markersize=2)
axes[1].get_lines()[1].set(color='#00C4FF')

# Rendements annuels (boxplot)
df_daily['Returns'].dropna().groupby(df_daily['Year']).apply(list)
years_data = [df_daily.loc[df_daily['Year'] == y, 'Returns'].dropna().values
              for y in sorted(df_daily['Year'].unique())]
bp = axes[2].boxplot(years_data, labels=sorted(df_daily['Year'].unique()),
                     patch_artist=True,
                     boxprops=dict(facecolor='#F7931A', alpha=0.6))
axes[2].set_title('Distribution par Année')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../plots/02_distribution_rendements.png', dpi=150, bbox_inches='tight')
plt.show()

# Stats descriptives des rendements
print('=== Statistiques des rendements ===')
print(f'Skewness    : {r.skew():.4f}')
print(f'Kurtosis    : {r.kurtosis():.4f}')
print(f'Sharpe (ann): {r.mean() / r.std() * np.sqrt(365):.4f}')

In [ ]:
# ── 3.3 Analyse par période / saisonnalité ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Saisonnalité & Patterns Temporels', fontsize=15, color='#F7931A')

# Prix moyen par mois
monthly = df_daily.groupby('Month')['Close'].median()
axes[0,0].bar(monthly.index, monthly.values, color=PALETTE[:12])
axes[0,0].set_xticks(range(1, 13))
axes[0,0].set_xticklabels(['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc'])
axes[0,0].set_title('Prix médian par Mois (tous ans)')

# Prix moyen par jour de semaine
weekly = df_daily.groupby('DayOfWeek')['Returns'].mean()
colors_w = ['#4ECDC4' if v >= 0 else '#FF6B6B' for v in weekly.values]
axes[0,1].bar(['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'], weekly.values, color=colors_w)
axes[0,1].set_title('Rendement moyen par Jour de la semaine')
axes[0,1].axhline(0, color='white', lw=0.5)

# Volatilité annuelle
yearly_vol = df_daily.groupby('Year')['Returns'].std() * np.sqrt(365) * 100
axes[1,0].plot(yearly_vol.index, yearly_vol.values, 'o-', color='#F7931A', lw=2)
axes[1,0].fill_between(yearly_vol.index, 0, yearly_vol.values, alpha=0.2, color='#F7931A')
axes[1,0].set_title('Volatilité Annualisée (%)')

# Heatmap rendements mensuels par année
pivot = df_daily.pivot_table(values='Returns', index='Year', columns='Month', aggfunc='mean')
pivot.columns = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
sns.heatmap(pivot, ax=axes[1,1], cmap='RdYlGn', center=0, annot=True, fmt='.3f', linewidths=0.5)
axes[1,1].set_title('Rendement moyen mensuel par Année')

plt.tight_layout()
plt.savefig('../plots/03_saisonnalite.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Stationnarité & Décomposition

In [ ]:
# ── Test de stationnarité (ADF & KPSS) ───────────────────────────────────────
def test_stationarity(series, name='Série'):
    """Effectue les tests ADF et KPSS et affiche les résultats."""
    print(f'\n{'='*50}')
    print(f'Test de Stationnarité : {name}')
    print('='*50)
    
    # Test ADF
    adf_result = adfuller(series.dropna(), autolag='AIC')
    print(f'\nTest ADF (H0 : non-stationnaire)')
    print(f'  Statistique : {adf_result[0]:.4f}')
    print(f'  p-value     : {adf_result[1]:.4f}', 
          '→ ✅ STATIONNAIRE' if adf_result[1] < 0.05 else '→ ❌ NON-STATIONNAIRE')
    
    # Test KPSS
    kpss_result = kpss(series.dropna(), regression='ct')
    print(f'\nTest KPSS (H0 : stationnaire)')
    print(f'  Statistique : {kpss_result[0]:.4f}')
    print(f'  p-value     : {kpss_result[1]:.4f}',
          '→ ✅ STATIONNAIRE' if kpss_result[1] > 0.05 else '→ ❌ NON-STATIONNAIRE')

# Séries à tester
test_stationarity(df_daily['Close'], 'Prix Close')
test_stationarity(df_daily['Log_Returns'].dropna(), 'Log-Rendements')
test_stationarity(np.log(df_daily['Close']).diff().dropna(), '1ère différence Log(Close)')

In [ ]:
# ── ACF / PACF ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('ACF & PACF — Prix Close vs Log-Rendements', fontsize=14, color='#F7931A')

# Série originale
plot_acf(df_daily['Close'].dropna(), lags=50, ax=axes[0,0], title='ACF — Prix Close', color='#F7931A')
plot_pacf(df_daily['Close'].dropna(), lags=50, ax=axes[0,1], title='PACF — Prix Close', color='#F7931A')

# Log-rendements (stationnaires)
plot_acf(df_daily['Log_Returns'].dropna(), lags=50, ax=axes[1,0], title='ACF — Log-Rendements', color='#00C4FF')
plot_pacf(df_daily['Log_Returns'].dropna(), lags=50, ax=axes[1,1], title='PACF — Log-Rendements', color='#00C4FF')

plt.tight_layout()
plt.savefig('../plots/04_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Décomposition de la série ─────────────────────────────────────────────────
# Sous-ensemble récent (2018–2021 pour lisibilité)
df_recent = df_daily.loc['2018':'2021']['Close']

decomp = seasonal_decompose(df_recent, model='multiplicative', period=365)

fig, axes = plt.subplots(4, 1, figsize=(16, 12))
fig.suptitle('Décomposition Multiplicative — Bitcoin (2018–2021)', fontsize=14, color='#F7931A')

decomp.observed.plot(ax=axes[0], color='#F7931A', title='Série Observée')
decomp.trend.plot(ax=axes[1], color='#00C4FF', title='Tendance')
decomp.seasonal.plot(ax=axes[2], color='#4ECDC4', title='Saisonnalité')
decomp.resid.plot(ax=axes[3], color='#FF6B6B', title='Résidus')

for ax in axes:
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../plots/05_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Modèle ARIMA

**ARIMA(p, d, q)** : Auto-Regressive Integrated Moving Average
- `p` : ordre AR (autorégressif)
- `d` : degré de différenciation
- `q` : ordre MA (moyenne mobile)

In [ ]:
# ── Préparation train/test ────────────────────────────────────────────────────
# On travaille sur le log du prix pour stabiliser la variance
series_log = np.log(df_daily['Close'].dropna())

# Split : 80% train, 20% test
n_test   = 90  # 90 jours de prévision
train_log = series_log.iloc[:-n_test]
test_log  = series_log.iloc[-n_test:]

train_close = df_daily['Close'].dropna().iloc[:-n_test]
test_close  = df_daily['Close'].dropna().iloc[-n_test:]

print(f'Train : {train_log.index[0].date()} → {train_log.index[-1].date()} ({len(train_log)} jours)')
print(f'Test  : {test_log.index[0].date()}  → {test_log.index[-1].date()}  ({len(test_log)} jours)')

In [ ]:
# ── Auto-ARIMA (recherche automatique des paramètres optimaux) ────────────────
print('🔍 Recherche automatique des paramètres ARIMA...')
auto_arima = pm.auto_arima(
    train_log,
    start_p=0, max_p=5,
    start_q=0, max_q=5,
    d=1,
    seasonal=False,
    stepwise=True,
    information_criterion='aic',
    suppress_warnings=True,
    error_action='ignore'
)
print(f'\n✅ Meilleur modèle ARIMA : {auto_arima.order}')
print(f'   AIC : {auto_arima.aic():.2f}')
print(auto_arima.summary())

In [ ]:
# ── Entraînement ARIMA ────────────────────────────────────────────────────────
p, d, q = auto_arima.order
arima_model = ARIMA(train_log, order=(p, d, q))
arima_fit   = arima_model.fit()

# Prévisions
arima_forecast_log = arima_fit.forecast(steps=n_test)
arima_forecast     = np.exp(arima_forecast_log)  # retour à l'échelle originale
arima_conf         = arima_fit.get_forecast(n_test).conf_int()
arima_lower        = np.exp(arima_conf.iloc[:, 0])
arima_upper        = np.exp(arima_conf.iloc[:, 1])

# Métriques
arima_mae  = mean_absolute_error(test_close, arima_forecast)
arima_rmse = np.sqrt(mean_squared_error(test_close, arima_forecast))
arima_mape = np.mean(np.abs((test_close.values - arima_forecast.values) / test_close.values)) * 100

print(f'\n=== Métriques ARIMA({p},{d},{q}) ===')
print(f'MAE  : ${arima_mae:,.0f}')
print(f'RMSE : ${arima_rmse:,.0f}')
print(f'MAPE : {arima_mape:.2f}%')

In [ ]:
# ── Visualisation ARIMA ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(train_close.iloc[-180:].index, train_close.iloc[-180:], 
        color='#F7931A', lw=1.5, label='Données historiques')
ax.plot(test_close.index, test_close, color='white', lw=1.5, label='Réel (test)', alpha=0.8)
ax.plot(test_close.index, arima_forecast, color='#00C4FF', lw=2, label='Prévision ARIMA', linestyle='--')
ax.fill_between(test_close.index, arima_lower, arima_upper, alpha=0.2, color='#00C4FF', label='IC 95%')
ax.set_title(f'ARIMA({p},{d},{q}) — MAE: ${arima_mae:,.0f} | MAPE: {arima_mape:.1f}%', fontsize=13)
ax.legend()
ax.set_ylabel('Prix BTC (USD)')
plt.tight_layout()
plt.savefig('../plots/06_arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Modèle SARIMA

**SARIMA(p,d,q)(P,D,Q,s)** : version saisonnière d'ARIMA
- `s` : période saisonnière (7 = hebdomadaire)

In [ ]:
# ── Auto-SARIMA ────────────────────────────────────────────────────────────────
print('🔍 Recherche paramètres SARIMA (saisonnalité hebdomadaire)...')
auto_sarima = pm.auto_arima(
    train_log,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=1, D=1,
    seasonal=True, m=7,
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    stepwise=True,
    information_criterion='aic',
    suppress_warnings=True,
    error_action='ignore'
)
print(f'\n✅ Meilleur modèle SARIMA : {auto_sarima.order} x {auto_sarima.seasonal_order}')
print(f'   AIC : {auto_sarima.aic():.2f}')

In [ ]:
# ── Entraînement SARIMA ───────────────────────────────────────────────────────
sarima_model = SARIMAX(
    train_log,
    order=auto_sarima.order,
    seasonal_order=auto_sarima.seasonal_order
)
sarima_fit = sarima_model.fit(disp=False)

sarima_forecast_log = sarima_fit.forecast(steps=n_test)
sarima_forecast     = np.exp(sarima_forecast_log)
sarima_conf         = sarima_fit.get_forecast(n_test).conf_int()
sarima_lower        = np.exp(sarima_conf.iloc[:, 0])
sarima_upper        = np.exp(sarima_conf.iloc[:, 1])

sarima_mae  = mean_absolute_error(test_close, sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(test_close, sarima_forecast))
sarima_mape = np.mean(np.abs((test_close.values - sarima_forecast.values) / test_close.values)) * 100

print(f'\n=== Métriques SARIMA ===')
print(f'MAE  : ${sarima_mae:,.0f}')
print(f'RMSE : ${sarima_rmse:,.0f}')
print(f'MAPE : {sarima_mape:.2f}%')

---
## 7. Modèle Prophet (Facebook)

**Prophet** est conçu pour des séries temporelles avec :
- Tendances non-linéaires
- Saisonnalités multiples (hebdo, annuelle)
- Événements spéciaux (holidays)
- Données manquantes et outliers

In [ ]:
# ── Préparation données Prophet (format ds/y obligatoire) ─────────────────────
df_prophet = df_daily['Close'].dropna().reset_index()
df_prophet.columns = ['ds', 'y']
df_prophet['y'] = np.log(df_prophet['y'])  # log pour stabiliser

train_prophet = df_prophet.iloc[:-n_test]
test_prophet  = df_prophet.iloc[-n_test:]

# ── Construction du modèle ────────────────────────────────────────────────────
prophet_model = Prophet(
    changepoint_prior_scale=0.3,    # flexibilité de la tendance
    seasonality_prior_scale=10,     # force de la saisonnalité
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)

# Saisonnalité mensuelle personnalisée
prophet_model.add_seasonality(name='monthly', period=30.5, fourier_order=5)

prophet_model.fit(train_prophet)
print('✅ Prophet entraîné')

In [ ]:
# ── Prévisions Prophet ────────────────────────────────────────────────────────
future   = prophet_model.make_future_dataframe(periods=n_test + 30)
forecast = prophet_model.predict(future)

# Prévisions sur la période de test
prophet_pred = np.exp(forecast.iloc[-n_test-30:-30]['yhat'].values)
prophet_lower = np.exp(forecast.iloc[-n_test-30:-30]['yhat_lower'].values)
prophet_upper = np.exp(forecast.iloc[-n_test-30:-30]['yhat_upper'].values)

actual_test = np.exp(test_prophet['y'].values)

prophet_mae  = mean_absolute_error(actual_test, prophet_pred)
prophet_rmse = np.sqrt(mean_squared_error(actual_test, prophet_pred))
prophet_mape = np.mean(np.abs((actual_test - prophet_pred) / actual_test)) * 100

print(f'\n=== Métriques Prophet ===')
print(f'MAE  : ${prophet_mae:,.0f}')
print(f'RMSE : ${prophet_rmse:,.0f}')
print(f'MAPE : {prophet_mape:.2f}%')

In [ ]:
# ── Visualisation composantes Prophet ─────────────────────────────────────────
fig = prophet_model.plot_components(forecast)
fig.suptitle('Décomposition Prophet : Tendance + Saisonnalités', color='#F7931A', fontsize=13)
plt.savefig('../plots/07_prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

# Prévision 30 jours supplémentaires
fig2 = prophet_model.plot(forecast)
fig2.suptitle('Prévisions Prophet — Bitcoin', color='#F7931A', fontsize=13)
plt.savefig('../plots/07b_prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Modèle XGBoost

**XGBoost** utilise des features temporelles pour prédire le prix suivant. 
On crée des features de lag et des indicateurs techniques.

In [ ]:
# ── Feature Engineering pour XGBoost ─────────────────────────────────────────
def create_features_xgb(df, target='Close', lags=[1,2,3,7,14,21,30]):
    """Crée des features de lag, rolling stats et indicateurs temporels."""
    X = pd.DataFrame(index=df.index)
    
    # Features de lag
    for lag in lags:
        X[f'lag_{lag}'] = df[target].shift(lag)
    
    # Rolling statistics
    for w in [7, 14, 30]:
        X[f'rolling_mean_{w}'] = df[target].rolling(w).mean()
        X[f'rolling_std_{w}']  = df[target].rolling(w).std()
        X[f'rolling_max_{w}']  = df[target].rolling(w).max()
        X[f'rolling_min_{w}']  = df[target].rolling(w).min()
    
    # Indicateurs techniques
    X['RSI']        = df.get('RSI', pd.Series(index=df.index, dtype=float))
    X['Volatility'] = df.get('Volatility_30', pd.Series(index=df.index, dtype=float))
    X['Volume']     = df.get('Volume_(Currency)', pd.Series(index=df.index, dtype=float))
    
    # Features temporelles
    X['month']      = df.index.month
    X['dayofweek']  = df.index.dayofweek
    X['quarter']    = df.index.quarter
    X['year']       = df.index.year
    
    y = df[target]
    return X, y

X_all, y_all = create_features_xgb(df_daily)
X_all.dropna(inplace=True)
y_all = y_all[X_all.index]

# Split temporel
X_train_xgb = X_all.iloc[:-n_test]
X_test_xgb  = X_all.iloc[-n_test:]
y_train_xgb = y_all.iloc[:-n_test]
y_test_xgb  = y_all.iloc[-n_test:]

print(f'Features : {X_train_xgb.shape[1]}')
print(f'Train    : {X_train_xgb.shape[0]} observations')
print(f'Test     : {X_test_xgb.shape[0]} observations')

In [ ]:
# ── Entraînement XGBoost ──────────────────────────────────────────────────────
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=50,
    verbosity=0
)

xgb_model.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_test_xgb, y_test_xgb)],
    verbose=False
)

xgb_pred = xgb_model.predict(X_test_xgb)

xgb_mae  = mean_absolute_error(y_test_xgb, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test_xgb, xgb_pred))
xgb_mape = np.mean(np.abs((y_test_xgb.values - xgb_pred) / y_test_xgb.values)) * 100

print(f'\n=== Métriques XGBoost ===')
print(f'MAE  : ${xgb_mae:,.0f}')
print(f'RMSE : ${xgb_rmse:,.0f}')
print(f'MAPE : {xgb_mape:.2f}%')

In [ ]:
# ── Feature Importance XGBoost ────────────────────────────────────────────────
feat_imp = pd.Series(xgb_model.feature_importances_, index=X_train_xgb.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.tail(20).plot(kind='barh', ax=ax, color='#F7931A', alpha=0.8)
ax.set_title('XGBoost — Feature Importance (Top 20)', fontsize=13, color='#F7931A')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../plots/08_xgboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Modèle LSTM

**LSTM** (Long Short-Term Memory) : réseau de neurones récurrent capable de mémoriser des dépendances à long terme dans les séries temporelles.

In [ ]:
# ── Préparation données LSTM ──────────────────────────────────────────────────
# Normalisation [0, 1]
scaler   = MinMaxScaler()
close_sc = scaler.fit_transform(df_daily[['Close']].dropna())

def create_sequences(data, seq_len=60):
    """Crée des séquences glissantes pour le LSTM."""
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i-seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

SEQ_LEN = 60  # 60 jours d'historique pour prédire J+1
X_seq, y_seq = create_sequences(close_sc, SEQ_LEN)
X_seq = X_seq.reshape(-1, SEQ_LEN, 1)

# Split
split     = len(X_seq) - n_test
X_tr, X_te = X_seq[:split], X_seq[split:]
y_tr, y_te = y_seq[:split], y_seq[split:]

print(f'X_train : {X_tr.shape}  |  X_test : {X_te.shape}')

In [ ]:
# ── Architecture LSTM ─────────────────────────────────────────────────────────
tf.random.set_seed(42)

lstm_model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.1),
    Dense(16, activation='relu'),
    Dense(1)
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='huber',
    metrics=['mae']
)

lstm_model.summary()

In [ ]:
# ── Entraînement LSTM ─────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history = lstm_model.fit(
    X_tr, y_tr,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ── Évaluation LSTM ───────────────────────────────────────────────────────────
lstm_pred_sc = lstm_model.predict(X_te)
lstm_pred    = scaler.inverse_transform(lstm_pred_sc).flatten()
lstm_actual  = scaler.inverse_transform(y_te.reshape(-1, 1)).flatten()

lstm_mae  = mean_absolute_error(lstm_actual, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred))
lstm_mape = np.mean(np.abs((lstm_actual - lstm_pred) / lstm_actual)) * 100

print(f'\n=== Métriques LSTM ===')
print(f'MAE  : ${lstm_mae:,.0f}')
print(f'RMSE : ${lstm_rmse:,.0f}')
print(f'MAPE : {lstm_mape:.2f}%')

# Courbe d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], color='#F7931A', label='Train Loss')
axes[0].plot(history.history['val_loss'], color='#00C4FF', label='Val Loss')
axes[0].set_title('Learning Curves — LSTM')
axes[0].legend()

idx_test = df_daily['Close'].dropna().index[-n_test:]
axes[1].plot(idx_test, lstm_actual, color='white', lw=1.5, label='Réel')
axes[1].plot(idx_test, lstm_pred,   color='#F7931A', lw=2, linestyle='--', label='LSTM')
axes[1].set_title(f'LSTM — MAPE: {lstm_mape:.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('../plots/09_lstm_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Sauvegarde du modèle
lstm_model.save('../models/lstm_bitcoin.h5')
print('\n✅ Modèle LSTM sauvegardé')

---
## 10. Comparaison des Modèles & Insights

In [ ]:
# ── Tableau comparatif ────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Modèle': ['ARIMA', 'SARIMA', 'Prophet', 'XGBoost', 'LSTM'],
    'MAE ($)': [arima_mae, sarima_mae, prophet_mae, xgb_mae, lstm_mae],
    'RMSE ($)': [arima_rmse, sarima_rmse, prophet_rmse, xgb_rmse, lstm_rmse],
    'MAPE (%)': [arima_mape, sarima_mape, prophet_mape, xgb_mape, lstm_mape],
    'Type': ['Statistique', 'Statistique', 'Additif', 'ML', 'Deep Learning']
})

results = results.sort_values('MAPE (%)')
print('\n=== CLASSEMENT DES MODÈLES (par MAPE) ===')
print(results.to_string(index=False))

results.to_csv('../data/model_comparison.csv', index=False)
print('\n✅ Résultats exportés')

In [ ]:
# ── Visualisation comparative ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Comparaison Complète des Modèles de Prévision', fontsize=16, color='#F7931A')

idx_test = df_daily['Close'].dropna().index[-n_test:]
y_true   = df_daily['Close'].dropna().values[-n_test:]

colors_models = {'ARIMA': '#00C4FF', 'SARIMA': '#4ECDC4', 'Prophet': '#A8E6CF',
                 'XGBoost': '#FF6B6B', 'LSTM': '#F7931A'}

# Graphique principal : toutes les prévisions
ax = axes[0, 0]
ax.plot(idx_test, y_true, color='white', lw=2, label='Réel', zorder=5)
ax.plot(idx_test, arima_forecast.values,   color='#00C4FF', lw=1.5, linestyle='--', label='ARIMA')
ax.plot(idx_test, sarima_forecast.values,  color='#4ECDC4', lw=1.5, linestyle='--', label='SARIMA')
ax.plot(idx_test, prophet_pred,            color='#A8E6CF', lw=1.5, linestyle='-.',  label='Prophet')
ax.plot(idx_test, xgb_pred,               color='#FF6B6B', lw=1.5, linestyle=':',  label='XGBoost')
ax.plot(idx_test, lstm_pred,              color='#F7931A', lw=2,   linestyle='--', label='LSTM')
ax.set_title('Toutes les prévisions vs Réel')
ax.legend(fontsize=8)

# MAPE comparatif
ax2 = axes[0, 1]
mapes = results[['Modèle', 'MAPE (%)']].set_index('Modèle')['MAPE (%)']
colors_bar = [colors_models[m] for m in mapes.index]
bars = ax2.barh(mapes.index, mapes.values, color=colors_bar, alpha=0.85)
for bar, val in zip(bars, mapes.values):
    ax2.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
             va='center', color='white', fontsize=10)
ax2.set_title('MAPE par Modèle (plus bas = meilleur)')
ax2.set_xlabel('MAPE (%)')

# Erreurs absolues journalières
ax3 = axes[1, 0]
ax3.plot(idx_test, np.abs(y_true - arima_forecast.values),   color='#00C4FF', lw=1, label='ARIMA', alpha=0.7)
ax3.plot(idx_test, np.abs(y_true - xgb_pred),               color='#FF6B6B', lw=1, label='XGBoost', alpha=0.7)
ax3.plot(idx_test, np.abs(y_true - lstm_pred),              color='#F7931A', lw=2, label='LSTM', alpha=0.9)
ax3.set_title('Erreur Absolue Journalière ($)')
ax3.set_ylabel('|Réel - Prédit| ($)')
ax3.legend()

# Scatter : Réel vs Prédit (meilleur modèle)
best_idx  = results['MAPE (%)'].idxmin()
best_name = results.loc[best_idx, 'Modèle']
best_pred_map = {'ARIMA': arima_forecast.values, 'SARIMA': sarima_forecast.values,
                 'Prophet': prophet_pred, 'XGBoost': xgb_pred, 'LSTM': lstm_pred}
best_pred = best_pred_map[best_name]

ax4 = axes[1, 1]
ax4.scatter(y_true, best_pred, alpha=0.5, color='#F7931A', s=20)
lim = [min(y_true.min(), best_pred.min()), max(y_true.max(), best_pred.max())]
ax4.plot(lim, lim, 'w--', lw=1, label='Prédiction parfaite')
ax4.set_xlabel('Prix Réel ($)')
ax4.set_ylabel('Prix Prédit ($)')
ax4.set_title(f'Réel vs Prédit — {best_name} (meilleur modèle)')
ax4.legend()

plt.tight_layout()
plt.savefig('../plots/10_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Insights & Recommandations ────────────────────────────────────────────────
best_mape  = results['MAPE (%)'].min()
worst_mape = results['MAPE (%)'].max()

print('='*60)
print('    SYNTHÈSE & INSIGHTS — BITCOIN TIME SERIES')
print('='*60)
print(f'\n📊 Dataset')
print(f'   Période analysée  : {df_daily.index[0].date()} → {df_daily.index[-1].date()}')
print(f'   Observations      : {len(df_daily):,} jours')
print(f'   Prix Min / Max    : ${df_daily["Close"].min():,.0f} / ${df_daily["Close"].max():,.0f}')
print(f'   Volatilité ann.   : {df_daily["Returns"].std() * np.sqrt(365) * 100:.1f}%')

print(f'\n🏆 Meilleur modèle  : {best_name} (MAPE = {best_mape:.1f}%)')
print(f'   Modèle le moins bon : {results.sort_values("MAPE (%)").iloc[-1]["Modèle"]} (MAPE = {worst_mape:.1f}%)')

print(f'\n💡 Recommandations')
print(f'   1. XGBoost et LSTM exploitent mieux les patterns non-linéaires')
print(f'   2. ARIMA/SARIMA utiles pour baseline rapide et interprétabilité')
print(f'   3. Prophet excellent pour capturer la saisonnalité annuelle')
print(f'   4. Combiner les modèles (ensemble) améliorerait les performances')
print(f'   5. Le Bitcoin reste très difficile à prévoir (haute volatilité)')
print(f'\n✅ Export CSV nettoyé + résultats des modèles terminé')